# 00 — Installation and API overview

This notebook checks your environment and introduces the one pattern every
RocqiPath workflow follows. It runs without OpenSlide, libvips or any model.

**You will learn to**

1. check that Jupyter uses Python 3.10 or 3.11 and which workflows are ready;
2. call any workflow as `rp.<workflow>(inputs, output_dir, **settings)`;
3. inspect, change, save and reload settings;
4. read a `Result` and the `rocqipath.json` record every run writes.

## Installation

From the repository root, in a terminal:

```bash
python -m pip install -e ".[extraction,orb,stain,cellcount,viz]"
python -m pip install jupyterlab
```

Install only the extras you need (`rocqipath list` shows what each workflow
needs). OpenSlide and libvips are native libraries; if your system lacks them,
`python -m pip install openslide-bin "pyvips[binary]"` provides both.

In [ ]:
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    """Find the repository whether Jupyter starts at its root or in how_to_use/."""
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "rocqipath").is_dir():
            return candidate
    return here


PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "data"          # your slides (kept out of git)
RESULTS_ROOT = PROJECT_ROOT / "results"    # outputs (kept out of git)
DEMO_ROOT = PROJECT_ROOT / "notebook_demo_outputs"  # synthetic examples

import rocqipath as rp

print(f"RocqiPath {rp.__version__}")
print(f"Data    : {DATA_ROOT}")
print(f"Results : {RESULTS_ROOT}")

In [ ]:
import platform
import sys

print(f"Python {platform.python_version()} ({sys.executable})")
if sys.version_info[:2] not in {(3, 10), (3, 11)}:
    print("WARNING: use a 64-bit Python 3.10 or 3.11 environment.")

from rocqipath.extras import missing_modules

for workflow in rp.list_workflows():
    missing = missing_modules(workflow.extra)
    state = "ready" if not missing else f"install rocqipath[{workflow.extra}]"
    print(f"{workflow.name:24s} {state:32s} {workflow.summary}")

## One pattern for every workflow

```python
result = rp.<workflow>(inputs, output_dir, *, config=None, **settings)
```

`settings` are fields of the workflow's config class. Every field is
documented; `help()` shows them all.

In [ ]:
help(rp.count_cells)

## Settings are typed configs

Create a config, derive variants with `replace`, and save it with your
results. Unknown names are rejected with a suggestion, and nested backend
settings use a double underscore.

In [ ]:
align_cfg = rp.AlignConfig(backend="orb", reference_name="he", moving_name="cd8", qc_enabled=True)
align_cfg = align_cfg.replace(target_magnification=20.0, orb__ransac_threshold=10.0)

for label, value in align_cfg.describe()[:8]:
    print(f"{label:32s} {value}")

try:
    align_cfg.replace(bakend="valis")
except TypeError as error:
    print("\n", error)

In [ ]:
import json

saved = json.dumps(align_cfg.to_dict(), indent=2)
restored = rp.AlignConfig.from_dict(json.loads(saved))
assert restored == align_cfg
print("Settings round-trip through JSON. TOML works too: rp.AlignConfig.from_toml('align.toml')")

## Results and the run record

Each run returns a `Result` listing the files it produced, and writes the
same information to `output_dir/rocqipath.json`. That record is what lets a
later workflow take an output folder as its input. Here a tiny synthetic
image is counted to show both.

In [ ]:
import numpy as np
from PIL import Image

image = np.full((256, 256, 3), (175, 185, 215), dtype=np.uint8)
yy, xx = np.mgrid[0:256, 0:256]
for cy, cx in [(60, 60), (60, 190), (190, 60), (190, 190), (128, 128)]:
    distance = np.sqrt((yy - cy) ** 2 + (xx - cx) ** 2)
    cell = distance <= 9
    image[cell] = (np.array([120, 70, 30]) * (0.55 + 0.6 * distance[cell] / 9)[:, None]).astype(np.uint8)
demo_image = DEMO_ROOT / "overview" / "five_cells.tif"
demo_image.parent.mkdir(parents=True, exist_ok=True)
Image.fromarray(image).save(demo_image)

result = rp.count_cells(demo_image, DEMO_ROOT / "overview" / "counts",
                        label="demo", source_magnification=20, patch_size=256, min_cell_area=20)
print("summary :", result.summary["results"][0]["total_positive"], "positive cells")
for item in result:
    print("item    :", item.role, item.path.name)
print("record  :", result.manifest_path)

## Checklist for reproducible work

- Keep raw slides read-only and outside the repository.
- Give each workflow its own output folder and let RocqiPath create the rest.
- Use physical magnifications (`20.0`) everywhere; set `source_magnification`
  only when a file has no objective metadata (notebook 01).
- Start with dry runs and small subsets; keep long cells behind `RUN_*` switches.
- The exact settings of every run are saved in `rocqipath.json`.

Continue with **01** to open real slides.